# Import Basic Libraries and Modules

In [1]:
# Loading Libraries
import warnings
warnings.filterwarnings('ignore')

import gc
gc.collect()

import pandas as pd
import numpy as np
import pickle
import os
import time

from rdkit.Chem import Descriptors
from rdkit.Chem import AllChem, Descriptors3D

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, cohen_kappa_score
from sklearn.inspection import PartialDependenceDisplay
from sklearn.utils import resample

import shap

import matplotlib.pyplot as plt
import seaborn as sns



# Inputs & Parameters

In [2]:

# File paths and other parameters
# File paths and other parameters
Nbins= 2
property= 'Activity'
target_style= 'NG'
featurizer_style= 'All_Featurizers_'+target_style
ml_model_style= 'All_Models_'+target_style
importance_style= 'SHAP'

# Names of files that contain SMILES, composition, and target property 
smiles_filepath = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Data\Data_Combined\{property}\InVitro_SMILES_All.xlsx"
smiles_sheet_name= "SMILES"

target_filepath = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Data\Data_Combined\{property}\InVitro_{property}_All.xlsx"
target_sheet_name= f"{Nbins} bins"

dataX_filepath = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{property}_pickle_files\dataX_dict_all_{featurizer_style}.pkl"
datay_filepath = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{property}_pickle_files\datay_all_{Nbins}bins_{target_style}.pkl"
scaffolds_filepath = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\{property}_pickle_files\Scaffolds.pkl"


output_file = rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\Feature_Importance\{property}_Feature_Importance_{importance_style}.xlsx"



# Implementation >>>>

# Load pickle files

In [3]:
# Load datasets

smiles_df = pd.read_excel(smiles_filepath, sheet_name= smiles_sheet_name)
smiles_list = smiles_df.iloc[:, 0].dropna().tolist()

target_df = pd.read_excel(target_filepath, sheet_name= target_sheet_name)

Nrows, Ncolumns = smiles_df.shape
Nconstituents= (Ncolumns-3)//2


with open(dataX_filepath, 'rb') as file:
    print('Reading feature variable')
    dataX_dict_all = pickle.load(file)
    
with open(datay_filepath, 'rb') as file1:
    print('Reading target variable')
    datay_all= pickle.load(file1)


Reading feature variable
Reading target variable


# Get names of RDKit descriptors

In [4]:
RDKit_descriptor_names = [name for name, _ in Descriptors._descList if name != "Ipc"]
RDKit_descriptor_names= RDKit_descriptor_names[:-1]

RDKit_descriptor_names_all = []
for i in range(1, 5):  # For constituents 1 to 4
    RDKit_descriptor_names_all.extend([f"{name} {i}" for name in RDKit_descriptor_names])


# Add extra descriptors
extra_descriptors = ['Composition 1', 'Composition 2', 'Composition 3', 
                     'Composition 4', 'RNA type', 'Lipid to RNA', 'Dosage']
RDKit_descriptor_names_all.extend(extra_descriptors)

print(len(RDKit_descriptor_names_all))

843


In [5]:
RDKit_3Ddescriptor_names= ['Asphericity', 'Eccentricity', 'InertialShapeFactor', 'NPR1', 'NPR2', 
                           'PBF', 'PMI1', 'PMI2', 'PMI3', 'RadiusOfGyration', 'SpherocityIndex']

RDKit_3Ddescriptor_names_all = []
for i in range(1, 5):  # For constituents 1 to 4
    RDKit_3Ddescriptor_names_all.extend([f"{name} {i}" for name in RDKit_3Ddescriptor_names])


# Add extra descriptors
extra_descriptors = ['Composition 1', 'Composition 2', 'Composition 3', 
                     'Composition 4', 'RNA type', 'Lipid to RNA', 'Dosage']
RDKit_3Ddescriptor_names_all.extend(extra_descriptors)

print(len(RDKit_3Ddescriptor_names_all))


51


# Generate/Get Molecular Scaffolds

In [6]:
def generate_scaffold(smiles):
    """Generate Bemis-Murcko scaffold from SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol) if mol else None


if os.path.exists(scaffolds_filepath):
    with open(scaffolds_filepath, 'rb') as file1:
        print('Reading scaffolds file')
        scaffolds= pickle.load(file1)
else:
    scaffolds = [generate_scaffold(sm) for sm in smiles_list]
    with open(scaffolds_filepath, 'wb') as file1:
        print('Writing scaffolds file')
        pickle.dump(scaffolds, file1)


Reading scaffolds file


# SHAP Importance

In [7]:

featurizer_name= 'RDKit_Descriptors_NG'
feature_names = RDKit_descriptor_names_all
# shap_values_file= rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\Feature_Importance\{property}_pickle_files\{property}_{importance_style}_Values_{featurizer_name}_{Nbins} bins.pkl"

dataX_all= dataX_dict_all[featurizer_name]
num_features = dataX_all.shape[1]
feature_id = [i for i in range(num_features)]

model = RandomForestClassifier(n_estimators= 100, max_depth= 10, min_samples_leaf= 4, min_samples_split= 10, random_state= 42, class_weight= 'balanced')
model.fit(dataX_all, datay_all)


explainer = shap.TreeExplainer(model)
shap_values_all = explainer.shap_values(dataX_all)


shap_values= shap_values_all[1]
# Compute average SHAP values (absolute importance)
shap_importance = np.abs(shap_values).mean(axis=0)
shap_importance_df = pd.DataFrame({'Feature ID': feature_id, 'Feature Name': feature_names, 'SHAP Importance': shap_importance})
shap_importance_sorted = shap_importance_df.sort_values(by='SHAP Importance', ascending=False)


    
with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:    
    shap_importance_df.to_excel(writer, sheet_name='Importance_'+featurizer_name, index=False)

print(f"Finished")



Finished


# Calculate Interaction Values

In [ ]:
# Select the top 100 features based on SHAP importance, to reduce memory usage, otherwise it's over 51 GB
top_k = 100
interactions_file= rf"C:\Users\grvkr\Box\Gaurav Kumar\Projects\SAR_NM\Codes\ML_Framework\Data_Combined\Feature_Importance\{property}_pickle_files\{property}_{importance_style}_Interactions_{featurizer_name}_{Nbins} bins.pkl"

top_features = shap_importance_df.nlargest(top_k, 'SHAP Importance')['Feature Name'].values
top_feature_indices = [feature_names.index(f) for f in top_features]  # Get corresponding indices

# Compute SHAP interaction values only for the selected features
interaction_values_all = explainer.shap_interaction_values(dataX_all[:, top_feature_indices].astype(np.float32))
# Keep only interactions for one class (e.g., class 1)
interaction_values = interaction_values_all[1]  # This reduces (n_samples, top_k, top_k, 2) to (n_samples, top_k, top_k)
del interaction_values_all  # Free up memory

with open(interactions_file, 'wb') as file1:
    print('Writing interactions file')
    pickle.dump(interaction_values, file1)


# Compute absolute interaction values for all pairs at once
interaction_values_abs = np.abs(interaction_values)
# Compute mean interaction importance in a single operation (FAST!)
interaction_means = interaction_values_abs.mean(axis=0)  # Shape: (top_k, top_k)

# Extract feature pairs and their interaction importances
i_idx, j_idx = np.triu_indices(top_k, k=1)  # Get upper triangle indices (avoid duplicate pairs)
interaction_list = list(zip(
    top_features[i_idx], 
    top_features[j_idx], 
    interaction_means[i_idx, j_idx]
))

# Convert to DataFrame
interaction_df_sorted = pd.DataFrame(interaction_list, columns=['Feature 1', 'Feature 2', 'Interaction Importance'])
interaction_df_sorted = interaction_df_sorted.sort_values(by='Interaction Importance', ascending=False)


# Save to Excel
with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:    
    interaction_df_sorted.to_excel(writer, sheet_name=f'Interactions_{featurizer_name}', index=False)

print(f"Finished computing interactions for top {top_k} features.")


In [ ]:

# Bar Plot
plt.figure(figsize=(10, 8))
sns.barplot(x='SHAP Importance', y='Feature Name', data=shap_importance_sorted[:20], palette='coolwarm')
plt.title('Top 20 Features by SHAP Importance')
plt.xlabel('Mean Absolute SHAP Value')
plt.ylabel('Feature Name')
plt.show()


In [ ]:


# Pivot DataFrame to create heatmap
interaction_pivot = interaction_df_sorted[0:100].pivot(index='Feature 1', columns='Feature 2', values='Interaction Importance')

plt.figure(figsize=(12, 10))
sns.heatmap(interaction_pivot, cmap='coolwarm', annot=False, fmt='.2f', linewidths=0.5)
plt.title('Shapley Interaction Index - Feature Interactions')
plt.show()


In [ ]:
# Create Network Graph
G = nx.Graph()

# Only show interactions above a threshold
threshold = 0.0005
filtered_interactions = interaction_df_sorted[interaction_df_sorted['Interaction Importance'] > threshold]

# Add nodes and edges
for _, row in filtered_interactions.iterrows():
    G.add_node(row['Feature 1'])
    G.add_node(row['Feature 2'])
    G.add_edge(row['Feature 1'], row['Feature 2'], weight=row['Interaction Importance'])

# Visualization
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42)
edges = G.edges(data=True)

nx.draw_networkx_nodes(G, pos, node_size=800, node_color='skyblue')
nx.draw_networkx_labels(G, pos, font_size=10)

edge_weights = [d['weight'] for _, _, d in edges]
nx.draw_networkx_edges(G, pos, edgelist=edges, width=edge_weights, edge_color=edge_weights, edge_cmap=plt.cm.coolwarm)

plt.title('Feature Interaction Network (Shapley Interaction Index)')
plt.show()


In [ ]:
shap.summary_plot(shap_values, dataX_all, feature_names=feature_names, plot_type="layered_violin", max_display=15)


# # If layere violin plot doesn't work
# # Define the indices of the descriptors you want to plot
# selected_indices = [11, 14, 3, 841, 44, 842, 66, 53, 59, 73, 57, 1, 0, 24, 22, 20, 28, 21, 83, 43, 47, 12, 104, 65, 191, 86, 198, 77, 82]

# # 1. Subset SHAP values: Select specific descriptors for all LNPs
# shap_values_selected = shap_values[:, selected_indices]

# # 2. Subset Data: Select specific descriptors from the dataset
# dataX_all_selected = dataX_all[:, selected_indices]

# # 3. Subset Descriptor Names: Select specific descriptor names
# feature_names_selected = [feature_names[i] for i in selected_indices]

# # 4. Generate SHAP Violin Plot (Layered Violin)
# import shap
# shap.plots.violin(
#     shap_values_selected, 
#     features=dataX_all_selected, 
#     feature_names=feature_names_selected, 
#     plot_type="layered_violin", 
#     layered_violin_max_num_bins=20, 
#     max_display=len(selected_indices)
# )


In [ ]:

shap.summary_plot(shap_values, dataX_all, feature_names=feature_names, plot_type="violin", max_display=15)

In [ ]:
shap.summary_plot(shap_values, features=dataX_all, feature_names=feature_names, max_display=15, cmap= "coolwarm")

In [ ]:
shap.dependence_plot("MaxPartialCharge 1", shap_values[1], dataX_testing, feature_names=RDKit_descriptor_names_all)

shap.dependence_plot("MaxPartialCharge 1", shap_values=shap_values[1], features=dataX_testing, feature_names=RDKit_descriptor_names_all, interaction_index="Dosage")

# Get Metrics using Specific Number of Top Features based on SHAP Importance

In [ ]:
# num_top_sheet_name= 'SHAP_Num_Top'
# featurizer_name= 'RDKit_Descriptors_NG'

# dataX_training= dataX_dict_training[featurizer_name]
# dataX_testing= dataX_dict_testing[featurizer_name]

# num_top_descriptors = [10, 25, 40, 45, 50, 55, 60, 75, 100, 200, 300, 400]

# # Initialize an empty dictionary to store the accuracies
# accuracy_dict = {}
# precision_dict = {}
# recall_dict = {}
# f1score_dict = {}

# for num_top in num_top_descriptors:
#     # Select top features based on feature importance
#     num_top_name= 'Top_'+ str(num_top)
#     top_indices = shap_importance_sorted.head(num_top)['Feature ID'].values
#     dataX_training_new = dataX_training[:, top_indices]
#     dataX_testing_new = dataX_testing[:, top_indices]

#     # Predictions using RF classification
#     predicted_class = ML_Model_RF(dataX_training_new, dataX_testing_new, datay_training, datay_testing)
#     true_class = datay_testing

#     classes = np.unique(true_class)
#     accuracy_per_class = {}
#     precision_per_class = {}
#     recall_per_class = {}
#     f1score_per_class = {}
#     for cls in classes:
#         # Binary classification: 1 for current class, 0 for all other classes
#         y_true_binary = [1 if y == cls else 0 for y in true_class]
#         y_pred_binary = [1 if y == cls else 0 for y in predicted_class]
#         accuracy = accuracy_score(y_true_binary, y_pred_binary)
#         precision = precision_score(y_true_binary, y_pred_binary)
#         recall = recall_score(y_true_binary, y_pred_binary)
#         f1score = f1_score(y_true_binary, y_pred_binary)
        
#         accuracy_per_class[cls] = accuracy
#         precision_per_class[cls] = precision
#         recall_per_class[cls] = recall
#         f1score_per_class[cls] = f1score
        
#     accuracy_dict[num_top_name] = accuracy_per_class
#     precision_dict[num_top_name] = precision_per_class
#     recall_dict[num_top_name] = recall_per_class
#     f1score_dict[num_top_name] = f1score_per_class


# # Prepare the data for the DataFrame
# rows = []

# for num_top in num_top_descriptors:
#     num_top_name= 'Top_'+ str(num_top)
#     # Initialize the row with model and featurizer names
#     row = {'Num Top Descriptors': num_top_name}

#     # Add accuracy for each class
#     for cls in classes:
#         row[f'Accuracy_{cls}'] = accuracy_dict[num_top_name].get(cls, None)
#     # Add precision for each class
#     for cls in classes:
#         row[f'Precision_{cls}'] = precision_dict[num_top_name].get(cls, None)
#     # Add recall for each class
#     for cls in classes:
#         row[f'Recall_{cls}'] = recall_dict[num_top_name].get(cls, None)
#     # Add f1 score for each class
#     for cls in classes:
#         row[f'F1 Score_{cls}'] = f1score_dict[num_top_name].get(cls, None)

#     # Append the row to the list
#     rows.append(row)

# # Convert to DataFrame
# results_df = pd.DataFrame(rows)

# # Write DataFrame to Excel
# # Check if the output file exists and read existing data if it does
# if os.path.exists(output_file):
#     with pd.ExcelFile(output_file, engine='openpyxl') as xls:
#         if num_top_sheet_name in xls.sheet_names:
#             existing_data_df = pd.read_excel(xls, sheet_name=num_top_sheet_name)
#             # Concatenate existing data with new data
#             combined_df = pd.concat([existing_data_df, results_df], ignore_index=True)
#         else:
#             combined_df = results_df
# else:
#     combined_df = results_df

# # Write the combined DataFrame to the Excel file
# with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
#     combined_df.to_excel(writer, sheet_name=num_top_sheet_name, index=False)

# print(f"Finished")


# H-statistics

In [ ]:

def calculate_h_statistic(model, X, feature_indices):
    """
    Calculate the H-statistic for interaction between two features.
    H = 1 - (independent_effect / combined_effect)
    """
    # Calculate Partial Dependence for individual features
    pdp_feature1 = PartialDependenceDisplay.from_estimator(model, X, [feature_indices[0]], grid_resolution=50, kind='average')
    pdp_feature2 = PartialDependenceDisplay.from_estimator(model, X, [feature_indices[1]], grid_resolution=50, kind='average')
    pdp_combined = PartialDependenceDisplay.from_estimator(model, X, [feature_indices], grid_resolution=50, kind='average')
    
    independent_effect = np.add(pdp_feature1.pd_results[0]['average'], pdp_feature2.pd_results[0]['average'])
    combined_effect = pdp_combined.pd_results[0]['average']

    # Calculate H-statistic
    h_statistic = 1 - np.var(combined_effect - independent_effect) / np.var(combined_effect)
    return h_statistic

# Calculate H-Statistic for Selected Feature Pairs
h_stats = []
for i in range(num_features):
    for j in range(i + 1, num_features):
        h_value = calculate_h_statistic(model, dataX_testing, [i, j])
        h_stats.append((feature_names[i], feature_names[j], h_value))

# Store Results in DataFrame
h_stat_df = pd.DataFrame(h_stats, columns=['Feature 1', 'Feature 2', 'H-Statistic'])
h_stat_df_sorted = h_stat_df.sort_values(by='H-Statistic', ascending=False)

# Save H-Statistic Results to Excel
output_file = "H_Statistic_Feature_Interactions.xlsx"
with pd.ExcelWriter(output_file, engine='openpyxl', mode='w') as writer:
    h_stat_df_sorted.to_excel(writer, sheet_name='Feature Interaction Strength', index=False)

print("✅ H-statistic calculated and saved.")


# Visualizing H-statistic results

In [ ]:


# Pivot DataFrame for Heatmap
h_stat_pivot = h_stat_df_sorted.pivot(index='Feature 1', columns='Feature 2', values='H-Statistic')

plt.figure(figsize=(12, 10))
sns.heatmap(h_stat_pivot, cmap='YlGnBu', annot=True, fmt='.2f', linewidths=0.5)
plt.title('H-Statistic - Feature Interaction Strength')
plt.show()


In [ ]:


# Select Top Interactions
top_interactions = h_stat_df_sorted.sort_values(by='H-Statistic', ascending=False).head(5)

for _, row in top_interactions.iterrows():
    feature_1 = row['Feature 1']
    feature_2 = row['Feature 2']
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=dataX_testing[:, RDKit_descriptor_names_all.index(feature_1)], 
                    y=dataX_testing[:, RDKit_descriptor_names_all.index(feature_2)], 
                    hue=datay_testing, palette='coolwarm')
    plt.title(f'Interaction: {feature_1} vs. {feature_2} (H = {row["H-Statistic"]:.2f})')
    plt.xlabel(feature_1)
    plt.ylabel(feature_2)
    plt.show()


# LIME Test

In [ ]:


# featurizer_style= 'RDKit_Descriptors_NG'
# # Iterate over the featurizer functions and ml model functions
# with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists= 'replace') as writer:
#     for ff in featurizer_map[featurizer_style]:
#         featurizer_name = ff.__name__  # Get the featurizer function name
    
#         dataX_training= dataX_dict_training[featurizer_name]
#         dataX_testing= dataX_dict_testing[featurizer_name]
    
#         model = RandomForestClassifier(n_estimators= 100, max_depth= 10, min_samples_leaf= 4, min_samples_split= 10, random_state= 42, class_weight= 'balanced')
#         model.fit(dataX_training, datay_training)
     
#         num_features = dataX_training.shape[1]
#         feature_id = [i for i in range(num_features)]
        
#         # Explain predictions
#         explainer = lime.lime_tabular.LimeTabularExplainer(dataX_training, 
#                                                    feature_names=RDKit_descriptor_names, 
#                                                    class_names=['0', '1'], 
#                                                    discretize_continuous=True)

# print(f"Finished")


# # Explain a single prediction
# instance_idx = 0
# exp = explainer.explain_instance(dataX_testing[instance_idx], model.predict_proba)
# exp.show_in_notebook()

# # # Save explanation
# # exp.save_to_file('lime_explanation.html')


# Write top 100 feature values to file

In [ ]:
num_estimators= 200
num_neighbors= 5
num_random= 42
num_top= 100

dataX_train= dataX_train_dict['RDKit_Descriptors_Featurizer']
datay_train= datay_train_dict['RDKit_Descriptors_Featurizer']
dataX_test= dataX_test_dict['RDKit_Descriptors_Featurizer']
datay_test= datay_test_dict['RDKit_Descriptors_Featurizer']

num_features = dataX_train.shape[1]
feature_id = [i for i in range(num_features)]
model = RandomForestClassifier(n_estimators= num_estimators, random_state= num_random)
model.fit(dataX_train, datay_train)
importances = model.feature_importances_
feature_importances = pd.DataFrame({
'Feature ID': feature_id,
'Feature Importance': importances
})
feature_importances = feature_importances.sort_values(by='Feature Importance', ascending=False)
top_indices = feature_importances.head(num_top)['Feature ID'].values

# Create a DataFrame for datay_test
df_datay_test = pd.DataFrame(datay_test, columns=['True Class'])
# Create a DataFrame for dataX_test[:, top_indices]
df_dataX_test_top = pd.DataFrame(dataX_test[:, top_indices], columns=[f'Feature {i+1}' for i in top_indices])
# Concatenate both DataFrames along columns
df_combined = pd.concat([df_datay_test, df_dataX_test_top], axis=1)

# Write the combined DataFrame to an Excel file
with pd.ExcelWriter(output_file, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_combined.to_excel(writer, sheet_name=f'Top_{num_top}_Features', index=False)


In [9]:
# !jupyter nbconvert --to script Main_SHAP.ipynb

[NbConvertApp] Converting notebook Main_SHAP.ipynb to script
[NbConvertApp] Writing 20859 bytes to Main_SHAP.py
